In [ ]:
# # !pip install -q -U transformers accelerate

# import json
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer

# # 1. Device Configuration
# # Automatically detects Kaggle's GPU (T4/P100) or falls back to CPU
# device = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"[System] Using device: {device.upper()}")

# # 2. Load Model & Tokenizer
# MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
# print(f"[System] Loading {MODEL_ID}...")

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
#     device_map="auto" if torch.cuda.is_available() else None
# )
# if device == "cpu":
#     model.to("cpu")

# # 3. Tool Definitions (OpenAI Schema Format)
# tools = [
#     {
#         "type": "function",
#         "function": {
#             "name": "write_expenditure",
#             "description": "Logs a financial purchase or farm expense.",
#             "parameters": {
#                 "type": "object",
#                 "properties": {
#                     "category": {
#                         "type": "string",
#                         "enum": ["feed", "medicine", "equipment", "infrastructure"]
#                     },
#                     "amount": {
#                         "type": "number",
#                         "description": "The cost parameter in Naira."
#                     },
#                     "description": {
#                         "type": "string",
#                         "description": "Brief English summary of the item."
#                     }
#                 },
#                 "required": ["category", "amount", "description"]
#             }
#         }
#     },
#     {
#         "type": "function",
#         "function": {
#             "name": "get_sensor_data",
#             "description": "Retrieves IoT telemetry data for a specific farm zone.",
#             "parameters": {
#                 "type": "object",
#                 "properties": {
#                     "sensor_type": {
#                         "type": "string", 
#                         "enum": ["temperature", "moisture", "water_level"]
#                     },
#                     "zone": {
#                         "type": "string",
#                         "description": "The farm zone extracted from the query."
#                     }
#                 },
#                 "required": ["sensor_type", "zone"]
#             }
#         }
#     },
#     {
#         "type": "function",
#         "function": {
#             "name": "get_health_log",
#             "description": "Retrieves the medical history for a specific animal.",
#             "parameters": {
#                 "type": "object",
#                 "properties": {
#                     "animal_id": {
#                         "type": "string",
#                         "description": "The numeric or string identifier of the animal."
#                     }
#                 },
#                 "required": ["animal_id"]
#             }
#         }
#     }
# ]

# # 4. Test Queries (Pidgin, Hausa, English)
# test_queries = [
#     "I just spend 45000 Naira buy bird feed.",
#     "Wane magani aka ba awaki mai lamba 204 a watan jiya?", 
#     "Check the moisture levels in the Barn A."
# ]

# system_prompt = "You are a localized farm management AI. Extract entities and trigger the appropriate function."

# print("\n" + "="*50)
# print("EXECUTING TOOL CALLING TESTS")
# print("="*50)

# # 5. Inference Execution Loop
# for query in test_queries:
#     print(f"\nUser Query: {query}")
    
#     messages = [
#         {"role": "system", "content": system_prompt},
#         {"role": "user", "content": query}
#     ]
    
#     # Qwen2.5 supports the OpenAI tools array directly in transformers via Jinja templates.
#     # This automatically formats the prompt to trigger JSON generation without requiring external libraries.
#     text_input = tokenizer.apply_chat_template(
#         messages,
#         tools=tools,
#         tokenize=False,
#         add_generation_prompt=True
#     )
    
#     model_inputs = tokenizer([text_input], return_tensors="pt").to(model.device)
    
#     with torch.no_grad():
#         generated_ids = model.generate(
#             **model_inputs,
#             max_new_tokens=256,
#             temperature=0.01, 
#             do_sample=False
#         )
    
#     generated_ids = [
#         output_ids[len(input_ids):] 
#         for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
#     ]
    
#     response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
#     print(f"Model Output:\n{response}")

In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install transformers accelerate

Looking in indexes: https://download.pytorch.org/whl/cpu


In [ ]:
# import time
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer

# # 1. Device Setup
# device = "cuda" if torch.cuda.is_available() else "cpu"
# dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# print(f"[System] Loading {MODEL_ID} on {device.upper()}...")
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     torch_dtype=dtype,
#     device_map="auto" if device == "cuda" else None
# )

# if device == "cpu":
#     model.to("cpu")

# # 2. Base System Prompts for Language Testing
# SYSTEM_PROMPTS = {
#     "1": ("Pidgin Mode", "You are a helpful farm assistant. Answer the user strictly in clear Nigerian Pidgin."),
#     "2": ("Hausa Mode", "Kai mataimakin manoma ne. Amsa duk tambayoyin amfani da harshen Hausa kadai."),
#     "3": ("Neutral / Auto-Detect Mode", "You are a localized farm assistant. Respond in whichever language the user addresses you in (English, Hausa, or Nigerian Pidgin).")
# }

# print("\nSelect evaluation mode:")
# print("1 - Force Nigerian Pidgin outputs")
# print("2 - Force Hausa outputs")
# print("3 - Auto-detect (Respond in user's language)")

# choice = input("\nEnter choice (1/2/3) [Default: 3]: ").strip()
# mode_name, selected_system_prompt = SYSTEM_PROMPTS.get(choice, SYSTEM_PROMPTS["3"])

# print(f"\n[Mode Locked]: {mode_name}")
# print("Type 'exit', 'quit', or 'reset' to manage the session.\n" + "="*50)

# # 3. Maintain Chat Context History
# conversation_history = [
#     {"role": "system", "content": selected_system_prompt}
# ]

# # 4. Interactive Conversation Loop
# while True:
#     user_input = input("\nYou > ").strip()
    
#     if not user_input:
#         continue
#     if user_input.lower() in ["exit", "quit"]:
#         print("Ending evaluation session.")
#         break
#     if user_input.lower() == "reset":
#         conversation_history = [{"role": "system", "content": selected_system_prompt}]
#         print("[Context cleared. Fresh session started.]")
#         continue

#     conversation_history.append({"role": "user", "content": user_input})

#     # Apply ChatML Template across full dialogue history
#     formatted_chat = tokenizer.apply_chat_template(
#         conversation_history,
#         tokenize=False,
#         add_generation_prompt=True
#     )

#     model_inputs = tokenizer([formatted_chat], return_tensors="pt").to(model.device)

#     # Time generation
#     start_time = time.time()
    
#     with torch.no_grad():
#         generated_ids = model.generate(
#             **model_inputs,
#             max_new_tokens=256,
#             temperature=0.3,  # Slight creative variance for natural conversation
#             do_sample=True,
#             top_p=0.9
#         )

#     # Trim input tokens from output
#     new_tokens = [
#         output_ids[len(input_ids):] 
#         for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
#     ]
    
#     response = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]
#     elapsed = time.time() - start_time
    
#     num_tokens = len(new_tokens[0])
#     tps = num_tokens / elapsed if elapsed > 0 else 0

#     print(f"\nQwen > {response}")
#     print(f"\n[Telemetry] {num_tokens} tokens generated in {elapsed:.2f}s ({tps:.2f} tokens/sec)")

#     # Save turn to history
#     conversation_history.append({"role": "assistant", "content": response})

In [ ]:
!ls /kaggle/input/datasets/matthewwisdom/synthetic-farm-data


In [2]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-4luuhham/unsloth_59304a486e22419a8449aafe46660dd4
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-4luuhham/unsloth_59304a486e22419a8449aafe46660dd4
  Resolved https://github.com/unslothai/unsloth.git to commit 57339b491a9902faf6517106cfaa69aac602cbc6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 45.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 111.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 69.4 MB/

In [ ]:
# import json
# import os

# INPUT_PATH = "/kaggle/input/datasets/matthewwisdom/synthetic-farm-data/synthetic_farm_data.jsonl"  # Update path if located under /kaggle/input/
# CLEANED_PATH = "cleaned_synthetic_farm_data.jsonl"

# cleaned_count = 0
# invalid_count = 0

# with open(INPUT_PATH, "r", encoding="utf-8") as infile, open(CLEANED_PATH, "w", encoding="utf-8") as outfile:
#     for line_num, line in enumerate(infile, 1):
#         line = line.strip()
#         if not line:
#             continue
#         try:
#             row = json.loads(line)
            
#             # Ensure "messages" exists and is a list
#             if "messages" not in row or not isinstance(row["messages"], list):
#                 invalid_count += 1
#                 continue
            
#             # Sanitize content fields across all roles
#             for msg in row["messages"]:
#                 if "content" in msg and not isinstance(msg["content"], str):
#                     # Convert dict or list content to a stringified JSON representation
#                     msg["content"] = json.dumps(msg["content"], ensure_ascii=False)
            
#             outfile.write(json.dumps(row, ensure_ascii=False) + "\n")
#             cleaned_count += 1

#         except json.JSONDecodeError:
#             invalid_count += 1

# print(f"Sanitization Complete: {cleaned_count} rows cleaned & saved to '{CLEANED_PATH}'.")
# print(f"Skipped {invalid_count} malformed rows.")

In [1]:
!ls

In [3]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import get_chat_template

# 1. Configuration
DATA_PATH = "/kaggle/input/datasets/matthewwisdom/agro-multiturn/synthetic_farm_data_multiturn.jsonl" # Updated to multi-turn dataset
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_SEQ_LENGTH = 2048

# 2. Load Base Model in 4-bit
print("Loading model and tokenizer...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# 3. Inject LoRA Adapters
print("Injecting LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# 4. Apply ChatML Template & Format Dataset
print("Formatting dataset...")
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
    mapping={"role": "role", "content": "content", "user": "user", "assistant": "assistant"}
)

def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        for messages in examples["messages"]
    ]
    return {"text": texts}

dataset = load_dataset("json", data_files=DATA_PATH, split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

# 5. Configure Training Arguments
print("Initializing SFTTrainer...")
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer, 
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        save_strategy="no",
        output_dir="outputs",
        average_tokens_across_devices=False, 
    ),
)

# 6. Execute Training
print("Starting training loop...")
try:
    trainer_stats = trainer.train()
except Exception as e:
    print(e)

# 7. Export to GGUF (4-bit Quantization)
print("Exporting model to 4-bit GGUF...")
model.save_pretrained_gguf("qwen_farm_agent", tokenizer, quantization_method="q4_k_m")

print("Pipeline complete.")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Injecting LoRA adapters...


Unsloth 2026.8.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.
[unsloth.chat_templates|WARNING]Unsloth: Will map <|im_end|> to EOS = <|im_end|>.


Formatting dataset...


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_unsloth_sentencepiece_temp/tokenizer_0okwshyd/tokenizer_config.json.


...

Starting training loop...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,942 | Num Epochs = 3 | Total steps = 741
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.723915


KeyboardInterrupt: 

In [ ]:
print("Exporting model to 4-bit GGUF...")
model.save_pretrained_gguf("qwen_farm_agent", tokenizer, quantization_method="q4_k_m")

In [ ]:
# !zip qwen_farm_agent.zip /kaggle/working/qwen_farm_agent

In [2]:
# !ls /kaggle/input/notebooks

matthewwisdom


In [1]:
!pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 MB 22.1 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.34-py3-none-linux_x86_64.whl size=20581022 sha256=9fb987e8bb610e085ac03f0736292dddf2c28b6acec93a68fc50af55920730e8
  Stored in directory: /root/.cache/pip/wheels/4a/10/e7/0eb9b120f1640844f33562a3964c5b18b67de1d66d3f9530e8
Successfully built llama-cpp-python


In [2]:
# import time
# import json
# import os
# from llama_cpp import Llama

# # 1. Configuration
# # Verify the exact path in your Kaggle file browser (right panel -> Input)
# MODEL_PATH = "/kaggle/input/notebooks/matthewwisdom/agroai/qwen_farm_agent_gguf/qwen2.5-3b-instruct.Q4_K_M.gguf"

# if not os.path.exists(MODEL_PATH):
#     raise FileNotFoundError(f"Cannot find model at {MODEL_PATH}. Check the 'Add Input' path.")

# SYSTEM_PROMPT = """You are a localized farm management AI. You have access to the following tools: [{"name": "write_expenditure", "parameters": ["category", "amount", "description"]}, {"name": "get_sensor_data", "parameters": ["sensor_type", "zone"]}, {"name": "get_health_log", "parameters": ["animal_id"]}, {"name": "query_knowledge_base", "parameters": ["search_query"]}]"""

# # 2. Load Model
# print(f"Loading {MODEL_PATH} into memory...")
# llm = Llama(
#     model_path=MODEL_PATH,
#     n_ctx=2048,
#     n_threads=4, 
#     verbose=False 
# )

# def format_chatml(system_prompt, user_query):
#     return f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_query}<|im_end|>\n<|im_start|>assistant\n"

# # 3. Test Cases (Hausa, Pidgin, Tool Constraints)
# test_queries = [
#     "Duba min yanayin Danshi na Zone 2.",                           # Hausa: Sensor Tool
#     "I buy 2 bags of goat feed for 45000 naira today.",             # Pidgin: Expenditure Tool
#     "Nawa ne kudin buhun masara guda biyu a Lokoja?",               # Hausa: RAG / Knowledge Base Tool
#     "Log say I buy medicine.",                                      # Pidgin: Missing parameters (Should ask for amount)
# ]

# print("\n" + "="*50)
# print("BATCH EVALUATION RUN")
# print("="*50)

# # 4. Execution Loop
# for query in test_queries:
#     print(f"\nUser > {query}")
#     formatted_prompt = format_chatml(SYSTEM_PROMPT, query)
    
#     start_time = time.time()
#     output = llm(
#         formatted_prompt,
#         max_tokens=256,
#         stop=["<|im_end|>"], 
#         temperature=0.1
#     )
    
#     response_text = output["choices"][0]["text"].strip()
#     elapsed = time.time() - start_time
#     tokens = output["usage"]["completion_tokens"]
    
#     print("Qwen > ", end="")
#     if response_text.startswith("[") and "function_name" in response_text:
#         try:
#             tool_call = json.loads(response_text)
#             print(f"[TOOL CALL] \n{json.dumps(tool_call, indent=2)}")
#         except json.JSONDecodeError:
#             print(f"[MALFORMED JSON] {response_text}")
#     else:
#         print(response_text)
        
#     print(f"[Telemetry: {tokens} tokens in {elapsed:.2f}s]")

In [3]:
# import time
# import json
# import os
# from llama_cpp import Llama
# from llama_cpp.llama_grammar import LlamaGrammar

# # 1. Configuration
# MODEL_PATH = "/kaggle/input/notebooks/matthewwisdom/agroai/qwen_farm_agent_gguf/qwen2.5-3b-instruct.Q4_K_M.gguf"

# if not os.path.exists(MODEL_PATH):
#     raise FileNotFoundError(f"Cannot find model at {MODEL_PATH}.")

# SYSTEM_PROMPT = """You are a localized farm management AI. You have access to the following tools: [{"name": "write_expenditure", "description": "Log a financial transaction into the expenditures table.", "parameters": ["category", "amount", "description"]}, {"name": "write_health_log", "description": "Record an animal's medical or physical event into the health_logs table.", "parameters": ["animal_id", "event_type", "notes"]}, {"name": "get_sensor_data", "description": "Retrieve readings from the telemetry_data table.", "parameters": ["node_id", "sensor_type"]}, {"name": "get_animal_record", "description": "Retrieve an animal's demographic and current status from the animals table.", "parameters": ["id"]}, {"name": "trigger_vision_reid", "description": "Execute the MegaDetector and MegaDescriptor pipeline on a locally saved image to identify an animal.", "parameters": ["image_filepath"]}, {"name": "query_knowledge_base", "description": "Search the RAG vector database for farming advice, disease treatment, or general agricultural knowledge.", "parameters": ["search_query"]}]"""

# # 2. Dynamic Grammar Generation via JSON Schema
# # This schema safely instructs the C++ backend to accept EITHER a tool array OR a string.
# master_schema = {
#     "anyOf": [
#         {
#             "type": "array",
#             "items": {
#                 "type": "object",
#                 "properties": {
#                     "function_name": {
#                         "type": "string",
#                         "enum": [
#                             "write_expenditure", 
#                             "write_health_log", 
#                             "get_sensor_data", 
#                             "get_animal_record", 
#                             "trigger_vision_reid", 
#                             "query_knowledge_base"
#                         ]
#                     },
#                     "arguments": {
#                         "type": "object"
#                     }
#                 },
#                 "required": ["function_name", "arguments"],
#                 "additionalProperties": False
#             }
#         },
#         {
#             "type": "string"
#         }
#     ]
# }

# # The library converts the JSON schema to a flawless GBNF graph automatically
# grammar = LlamaGrammar.from_json_schema(json.dumps(master_schema))

# # 3. Load Model
# print(f"Loading {MODEL_PATH} into memory...")
# llm = Llama(
#     model_path=MODEL_PATH,
#     n_ctx=2048,
#     n_threads=4, 
#     verbose=False 
# )

# def format_chatml(system_prompt, user_query):
#     return f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_query}<|im_end|>\n<|im_start|>assistant\n"

# # 4. Test Cases
# test_queries = [
#     "Duba min yanayin Danshi na Zone 2.",                           
#     "I buy 2 bags of goat feed for 45000 naira today.",             
#     "Nawa ne kudin buhun masara guda biyu a Lokoja?",               
#     "Log say I buy medicine.",                                      
# ]

# print("\n" + "="*50)
# print("BATCH EVALUATION RUN - JSON SCHEMA ENFORCED")
# print("="*50)

# # 5. Execution Loop
# for query in test_queries:
#     print(f"\nUser > {query}")
#     formatted_prompt = format_chatml(SYSTEM_PROMPT, query)
    
#     start_time = time.time()
#     output = llm(
#         formatted_prompt,
#         max_tokens=256,
#         stop=["<|im_end|>"], 
#         temperature=0.1,
#         grammar=grammar  # Dynamic schema applied here
#     )
    
#     response_text = output["choices"][0]["text"].strip()
#     elapsed = time.time() - start_time
#     tokens = output["usage"]["completion_tokens"]
    
#     print("Qwen > ", end="")
#     if response_text.startswith("[") and "function_name" in response_text:
#         try:
#             tool_call = json.loads(response_text)
#             print(f"[TOOL CALL] \n{json.dumps(tool_call, indent=2)}")
#         except json.JSONDecodeError:
#             print(f"[MALFORMED JSON] {response_text}")
#     else:
#         print(response_text)
        
#     print(f"[Telemetry: {tokens} tokens in {elapsed:.2f}s]")

In [4]:
import os
import time
import json
from llama_cpp import Llama
from llama_cpp.llama_grammar import LlamaGrammar

# ---------------------------------------------------------
# 1. Configuration & Model Loading
# ---------------------------------------------------------
# Adjust path if Unsloth named the file with a prefix like 'qwen_farm_agent-unsloth.Q4_K_M.gguf'
MODEL_PATH = "/kaggle/input/notebooks/matthewwisdom/agroai/qwen_farm_agent_gguf/qwen2.5-3b-instruct.Q4_K_M.gguf"

if not os.path.exists(MODEL_PATH):
    # Fallback check for alternative root directory naming
    if os.path.exists("qwen_farm_agent.Q4_K_M.gguf"):
        MODEL_PATH = "qwen_farm_agent.Q4_K_M.gguf"
    else:
        print(f"Warning: Could not auto-locate {MODEL_PATH}. Update MODEL_PATH manually.")

SYSTEM_PROMPT = """You are a localized farm management AI. You have access to the following tools: [{"name": "write_expenditure", "description": "Log a financial transaction into the expenditures table.", "parameters": ["category", "amount", "description"]}, {"name": "write_health_log", "description": "Record an animal's medical or physical event into the health_logs table.", "parameters": ["animal_id", "event_type", "notes"]}, {"name": "get_sensor_data", "description": "Retrieve readings from the telemetry_data table.", "parameters": ["node_id", "sensor_type"]}, {"name": "get_animal_record", "description": "Retrieve an animal's demographic and current status from the animals table.", "parameters": ["id"]}, {"name": "trigger_vision_reid", "description": "Execute the MegaDetector and MegaDescriptor pipeline on a locally saved image to identify an animal.", "parameters": ["image_filepath"]}, {"name": "query_knowledge_base", "description": "Search the RAG vector database for farming advice, disease treatment, or general agricultural knowledge.", "parameters": ["search_query"]}]"""

# ---------------------------------------------------------
# 2. Grammar Construction (JSON Schema AST)
# ---------------------------------------------------------
master_schema = {
    "anyOf": [
        {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "function_name": {
                        "type": "string",
                        "enum": [
                            "write_expenditure",
                            "write_health_log",
                            "get_sensor_data",
                            "get_animal_record",
                            "trigger_vision_reid",
                            "query_knowledge_base"
                        ]
                    },
                    "arguments": {
                        "type": "object"
                    }
                },
                "required": ["function_name", "arguments"],
                "additionalProperties": False
            }
        },
        {
            "type": "string"
        }
    ]
}

print("Compiling JSON Schema Grammar...")
grammar = LlamaGrammar.from_json_schema(json.dumps(master_schema))

print(f"Loading {MODEL_PATH} into memory...")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=2048,
    n_threads=4,
    verbose=False
)

# ---------------------------------------------------------
# 3. Test Cases (Single & Multi-Turn Scenarios)
# ---------------------------------------------------------
test_scenarios = [
    {
        "name": "Single-Turn Hausa Sensor Query",
        "history": [
            {"role": "user", "content": "Duba min yanayin Danshi na Zone 2."}
        ]
    },
    {
        "name": "Single-Turn Pidgin Expenditure",
        "history": [
            {"role": "user", "content": "I buy 2 bags of goat feed for 45000 naira today."}
        ]
    },
    {
        "name": "Single-Turn Incomplete Log (Needs Clarification)",
        "history": [
            {"role": "user", "content": "Log say I buy medicine."}
        ]
    },
    {
        "name": "Multi-Turn Context Resolution",
        "history": [
            {"role": "user", "content": "Log say I buy medicine."},
            {"role": "assistant", "content": "Wetin be di amount wey you pay for di medicine, and which animal get am?"},
            {"role": "user", "content": "Na 12000 naira for di goat GT-004."}
        ]
    },
    {
        "name": "Guardrail Check (Non-Farm Query)",
        "history": [
            {"role": "user", "content": "Who win Premier League match yesterday?"}
        ]
    }
]

def format_chatml(system_prompt, messages):
    formatted = f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
    for msg in messages:
        formatted += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"
    formatted += "<|im_start|>assistant\n"
    return formatted

# ---------------------------------------------------------
# 4. Benchmarking Execution
# ---------------------------------------------------------
def run_evaluation(mode_name, use_grammar):
    print("\n" + "="*60)
    print(f"  RUNNING BENCHMARK: {mode_name.upper()}")
    print("="*60)

    for idx, scenario in enumerate(test_scenarios, 1):
        print(f"\n--- Scenario {idx}: {scenario['name']} ---")
        
        # Display context turns if multi-turn
        if len(scenario["history"]) > 1:
            print("History Context:")
            for msg in scenario["history"][:-1]:
                print(f"  [{msg['role'].upper()}]: {msg['content']}")
            print(f"Latest Turn:\n  [USER]: {scenario['history'][-1]['content']}")
        else:
            print(f"User > {scenario['history'][0]['content']}")

        prompt = format_chatml(SYSTEM_PROMPT, scenario["history"])
        
        start_time = time.time()
        output = llm(
            prompt,
            max_tokens=256,
            stop=["<|im_end|>"],
            temperature=0.1,
            grammar=grammar if use_grammar else None
        )
        
        elapsed = time.time() - start_time
        response_text = output["choices"][0]["text"].strip()
        tokens = output["usage"]["completion_tokens"]

        print("Qwen > ", end="")
        if response_text.startswith("[") and "function_name" in response_text:
            try:
                tool_call = json.loads(response_text)
                print(f"[TOOL CALL]\n{json.dumps(tool_call, indent=2)}")
            except json.JSONDecodeError:
                print(f"[MALFORMED JSON] {response_text}")
        else:
            print(response_text)

        print(f"[Telemetry: {tokens} tokens in {elapsed:.2f}s | {tokens/elapsed:.1f} t/s]")

# ---------------------------------------------------------
# 5. Run Both Modes
# ---------------------------------------------------------
if __name__ == "__main__":
    # Pass 1: Unconstrained Weights Evaluation
    run_evaluation("Without Grammar (Raw Weights)", use_grammar=False)
    
    # Pass 2: Schema Constrained Evaluation
    run_evaluation("With Grammar (JSON Schema Constrained)", use_grammar=True)

Compiling JSON Schema Grammar...
Loading /kaggle/input/notebooks/matthewwisdom/agroai/qwen_farm_agent_gguf/qwen2.5-3b-instruct.Q4_K_M.gguf into memory...

  RUNNING BENCHMARK: WITHOUT GRAMMAR (RAW WEIGHTS)

--- Scenario 1: Single-Turn Hausa Sensor Query ---
User > Duba min yanayin Danshi na Zone 2.
Qwen > Na gode. But make you tell me, which sensor—like ‘node_id’—you want? And wetin be di type of sensor you want? Na shin ‘temperature’ or ‘humidity’?
[Telemetry: 43 tokens in 14.14s | 3.0 t/s]

--- Scenario 2: Single-Turn Pidgin Expenditure ---
User > I buy 2 bags of goat feed for 45000 naira today.
Qwen > [TOOL CALL]
[
  {
    "function_name": "write_expenditure",
    "arguments": {
      "category": "feed",
      "amount": 45000,
      "description": "2 bags goat feed"
    }
  }
]
[Telemetry: 40 tokens in 5.60s | 7.1 t/s]

--- Scenario 3: Single-Turn Incomplete Log (Needs Clarification) ---
User > Log say I buy medicine.
Qwen > Okay, but make you tell me, which kind of animal we talk a